# Batch selection and fitting

Discover all acquisition prefixes in `PMT_Data`, then run `Selection.ipynb` and `Fit.ipynb` for each one. Files such as `name-1.h5` through `name-5.h5` are treated as chunks of one acquisition.

In [ ]:
from pathlib import Path
import re

# ============================== DATASETS ==============================
# Add one dictionary per dataset. 'data_dir' contains the raw HDF5 files;
# 'analysis_dir' receives selection plots, cached dataframes, fits, and gain
# plots. Keep separate analysis directories when detector/filter settings differ.
batch_datasets = [
    {
        "name": "PMT_Data_no_bw_filter",
        "data_dir": Path("PMT_Data_no_bw_filter"),
        "analysis_dir": Path("plots/PMT_Data_no_bw_filter"),
    },
]

# ============================== FIT GRID ==============================
# Fit models to run: any subset of ['poisson', 'bellamy']. Poisson is faster
# and is a good first pass; Bellamy adds an under-amplified SPE component.
batch_fit_models = ["poisson"]
# Charge estimator: 'led_window' integrates the same trigger-relative window
# for every event and is preferred for LED calibration. 'full_waveform' is a
# useful systematic comparison. Example: ['led_window', 'full_waveform'].
batch_charge_methods = ["led_window"]
# Generate selections at these detected-peak SNR boundaries. Events with no
# qualifying peak or below a boundary are retained to preserve the pedestal.
# Start broad (for example [8, 10, 12, 15]) and compare fitted gain stability.
batch_cut_thresholds_snr = [8.0, 10.0, 12.0, 15.0]
# Select which generated names are actually fit; None fits all generated names.
# Names depend on mode: standard -> 'pulse_quality_above_snr15'; timing_only ->
# 'led_timing_above_snr15'; loose -> 'loose_peak_timing_above_snr15'. A named
# selection is authoritative and makes Fit.ipynb infer the corresponding mode.
batch_selection_names = ["led_timing_above_snr15"]
# Generate an uncut reference named 'no_peak_cuts'. To actually fit it, either
# set batch_selection_names=None or include 'no_peak_cuts' in that list. Useful
# for seeing how much peak-quality cuts move the fitted gain.
batch_include_no_peak_cuts = True
# Add learned central-98% width/rise/fall-time cuts. Leave False initially; turn
# on only after inspecting pulse shapes and confirming enough clean references.
batch_include_shape_cut = False
# Selection behavior above each batch_cut_thresholds_snr value:
#   'standard'                exactly 1 peak + LED timing (recommended for fast pulses)
#   'timing_only'             LED timing only; allows multiple detected peaks
#   'loose_peak_multiplicity' 1..batch_max_allowed_peaks + LED timing
# Detection itself always searches the full waveform; timing is only a later cut.
batch_selection_mode = "timing_only"
# Require exactly one peak when learning the expected LED time. True gives a
# cleaner reference for standard mode; False is useful for broad/oscillatory pulses.
batch_timing_reference_requires_single_peak = False
# Maximum accepted peak count in loose_peak_multiplicity mode; ignored otherwise.
# Example: 3 for mildly split pulses, 6 for strongly ringing waveforms.
batch_max_allowed_peaks = 6
# Minimum detected-peak SNR used only to learn the LED-time histogram mode. Choose
# a value with a clear timing cluster; 8 is permissive, 15 is cleaner.
batch_timing_reference_snr = 8.0
# Accept peaks within expected LED time +/- this many ns. Start near the measured
# timing spread: e.g. 5 ns for tight fast pulses, 10-15 ns for broader timing.
batch_peak_timing_tolerance_ns = 12.0

# =========================== PEAK DETECTION ===========================
# These settings change n_peaks/peak_time/SNR and therefore REQUIRE rerunning
# Selection.ipynb (run_selection=True) to regenerate cached *_df.pkl files.
# A local maximum must exceed this multiple of its waveform's baseline RMS.
# Start at 5; lower to 4 for missed small pulses, raise to 6-8 for noise peaks.
batch_peak_snr_threshold = 5.0
# Required vertical prominence above local surroundings, in baseline-RMS units.
# Start at 3; raise it if ringing/noise shoulders are counted as separate peaks.
# Use None to disable the prominence requirement.
batch_peak_prominence_snr = 3.0
# Minimum separation between counted peaks, in SAMPLES. At the measured 0.01562
# ns/sample, 128 samples ~= 2 ns. Raise it if one pulse is split into nearby peaks;
# lower it if genuinely distinct close pulses matter. None disables this condition.
batch_peak_distance_samples = 128
# Accepted peak FWHM range in SAMPLES. Here (64, 640) ~= (1, 10) ns at 0.01562
# ns/sample, covering observed 3-5 ns fast pulses and 8-9 ns filtered pulses.
# Use None to disable; verify the conversion if a run has a new sampling interval.
batch_peak_width_samples = (64, 640)

# =========================== DIAGNOSTIC PLOTS =========================
# Plot waveforms rejected by each active calibration selection. Set False to
# reduce raw-file reads and plotting time during production reruns.
batch_plot_rejected_waveforms = True
# Number of rejected traces overlaid per selection and random seed controlling
# which events are sampled. Example: 20 for quick checks, 100 for investigation.
batch_rejected_waveform_sample = 40
batch_rejected_waveform_seed = 12345
# Number of full traces in each example plot: no detected peak (pedestal-like)
# and detected peak with SNR above batch_example_high_snr_threshold.
batch_example_waveform_count = 10
# This changes only the high-SNR example plot, not detection or fit selections.
batch_example_high_snr_threshold = 15.0
# Fixed seed makes the same example waveforms appear on repeated runs.
batch_example_waveform_seed = 12345

# =========================== EXECUTION CONTROL ========================
# Save generated figures below each dataset's analysis_dir. False still displays
# figures interactively but does not write them to disk.
batch_save_plots = True
# True reloads raw waveforms and rebuilds cached features. REQUIRED after changing
# baseline/integration/peak-detection settings. False reuses *_df.pkl and is faster.
run_selection = True
# If True, abort rather than silently loading raw HDF5 when cached fit data are
# missing. Useful for fit-only production runs with run_selection=False.
batch_require_cached_fit_data = False
# Run charge-model fits after selection. Set False for selection/peak QA only.
run_fit = True
# True records a failed acquisition and continues; False stops at the first error.
continue_on_error = True

# Acquisition prefixes to process. None discovers every *.h5 prefix in data_dir.
# Example quick test: ['WA0089_850V_525MHz_led100']. Always test one run before
# launching the complete voltage sweep after changing peak-detection settings.
file_names = None

def acquisition_sort_key(name):
    voltage_match = re.search(r"_(\d+(?:\.\d+)?)V(?:_|$)", name)
    return (float(voltage_match.group(1)) if voltage_match else float("inf"), name)

def discover_file_names(data_dir):
    return sorted({
        re.sub(r"-\d+\.h5$", "", path.name)
        for path in data_dir.glob("*.h5")
    }, key=acquisition_sort_key)

print("Batch fit configuration")
print(f"  models: {batch_fit_models}")
print(f"  charge methods: {batch_charge_methods}")
print(f"  selections: {batch_selection_names}")
print(f"  selection mode: {batch_selection_mode}")
print(f"  SNR thresholds: {batch_cut_thresholds_snr}")

In [ ]:
import gc
import traceback
import pandas as pd
import matplotlib.pyplot as plt

batch_results = []
for dataset in batch_datasets:
    dataset_name = dataset["name"]
    data_dir = dataset["data_dir"]
    analysis_dir = dataset["analysis_dir"]
    batch_data_dir = data_dir
    batch_selection_output_dir = analysis_dir / "selection"
    batch_fit_output_dir = analysis_dir / "fit"
    batch_fit_inputs_dir = analysis_dir / "fit_data"
    batch_fit_results_dir = batch_fit_output_dir / "fit_results"
    batch_gain_output_dir = analysis_dir / "gain_calibration"
    dataset_file_names = file_names if file_names is not None else discover_file_names(data_dir)

    if not dataset_file_names:
        raise FileNotFoundError(f"No HDF5 acquisitions found in {data_dir.resolve()}")

    print(f"\n{'#' * 80}")
    print(f"Dataset: {dataset_name}")
    print(f"Found {len(dataset_file_names)} acquisitions in {data_dir}")
    print(f"All outputs will be written below {analysis_dir}")
    print(f"{'#' * 80}")

    for batch_index, batch_file_name in enumerate(dataset_file_names, start=1):
        print(f"\n{'=' * 80}")
        print(f"[{batch_index}/{len(dataset_file_names)}] {batch_file_name}")
        print(f"{'=' * 80}")
        status = {"dataset": dataset_name, "file_name": batch_file_name, "selection": "skipped", "fit": "skipped"}

        try:
            if run_selection:
                get_ipython().run_line_magic("run", "-i Selection.ipynb")
                status["selection"] = "ok"
            if run_fit:
                get_ipython().run_line_magic("run", "-i Fit.ipynb")
                status["fit"] = "ok"
        except Exception as exc:
            failed_stage = "selection" if status["selection"] != "ok" and run_selection else "fit"
            status[failed_stage] = f"failed: {type(exc).__name__}: {exc}"
            traceback.print_exc()
            if not continue_on_error:
                raise
        finally:
            batch_results.append(status)
            plt.close("all")
            gc.collect()

    dataset_successful_fits = sum(
        row["dataset"] == dataset_name and row["fit"] == "ok"
        for row in batch_results
    )
    if run_fit and dataset_successful_fits >= 2:
        print(f"\nCreating gain-calibration plots for {dataset_name} from {dataset_successful_fits} successful fits")
        get_ipython().run_line_magic("run", "-i Gain_calibration.ipynb")
    else:
        print(f"Skipping gain calibration for {dataset_name}: fewer than two fits completed successfully")

print("\nBatch summary")
batch_summary = pd.DataFrame(batch_results)
display(batch_summary)

successful_fits = batch_summary['fit'].eq('ok').sum()
print(f"Successful fit notebooks: {successful_fits}")